# TF-IDF baseline

Цель — получить сильную и понятную точку отсчёта для сравнения
с Transformer-моделями.

Основная метрика — **macro F1**, дополнительная — accuracy.
Официальный test на этапе разработки не используется.

In [1]:
import json
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

DATA_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")

X_train = train_df["text"]
y_train = train_df["category"]
X_val = val_df["text"]
y_val = val_df["category"]

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

Train: (7996, 2)
Validation: (2000, 2)


## Сравнение представлений текста

Проверяем два варианта при одинаковом классификаторе:

- униграммы — отдельные слова;
- униграммы и биграммы — отдельные слова и пары соседних слов.

In [2]:
def build_model(ngram_range: tuple[int, int]) -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(lowercase=True, min_df=2, ngram_range=ngram_range),
            ),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    )


def evaluate(model: Pipeline) -> tuple[dict, list[str]]:
    predictions = model.predict(X_val)

    metrics = {
        "accuracy": accuracy_score(y_val, predictions),
        "macro_f1": f1_score(y_val, predictions, average="macro"),
        "features": len(model.named_steps["tfidf"].vocabulary_),
    }

    return metrics, predictions

In [3]:
configurations = {"tfidf_unigrams": (1, 1), "tfidf_unigrams_bigrams": (1, 2)}

models = {}
predictions = {}
results = []

for name, ngram_range in configurations.items():
    model = build_model(ngram_range)

    started_at = time.perf_counter()
    model.fit(X_train, y_train)
    training_seconds = time.perf_counter() - started_at

    metrics, val_predictions = evaluate(model)

    models[name] = model
    predictions[name] = val_predictions
    results.append({"model": name, **metrics, "training_seconds": training_seconds})

results_df = (
    pd.DataFrame(results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(results_df)

,model,accuracy,macro_f1,features,training_seconds
0,tfidf_unigrams,0.865,0.861327,1357,0.580442
1,tfidf_unigrams_bigrams,0.844,0.839495,8859,2.236368


## Анализ ошибок лучшего baseline

Выводим небольшую случайную выборку ошибок, чтобы увидеть, какие
близкие банковские интенты модель путает.

In [4]:
best_name = results_df.loc[0, "model"]
best_model = models[best_name]
best_predictions = predictions[best_name]

probabilities = best_model.predict_proba(X_val)
confidence = probabilities.max(axis=1)

validation_predictions = val_df.assign(
    predicted_category=best_predictions, confidence=confidence
)

errors = validation_predictions.query("category != predicted_category")

display(
    errors.sample(n=min(12, len(errors)), random_state=42)[
        ["text", "category", "predicted_category", "confidence"]
    ]
)

print("Ошибок на validation:", len(errors))

,text,category,predicted_category,confidence
195,How do I withdraw money?,atm_support,declined_cash_withdrawal,0.086751
958,Where to check top ups pending?,pending_top_up,top_up_by_cash_or_cheque,0.233216
660,What caused my cash withdrawal to be declined?,declined_cash_withdrawal,pending_cash_withdrawal,0.200782
1033,What's the process for topping up by card?,topping_up_by_card,top_up_by_card_charge,0.212930
1462,Why does it still charge me even though I was ...,pending_cash_withdrawal,declined_cash_withdrawal,0.469776
1088,My foreign currency exchange has been applied ...,card_payment_wrong_exchange_rate,exchange_charge,0.225079
1555,why do you charge for transfers?,top_up_by_bank_transfer_charge,transfer_fee_charged,0.228603
342,What are the steps for making a transfer for ...,transfer_into_account,top_up_by_bank_transfer_charge,0.167037
1209,I cannot get my virtual card to function.,virtual_card_not_working,getting_virtual_card,0.280672
1848,what is the fee for a transfer into my account,top_up_by_bank_transfer_charge,transfer_fee_charged,0.237986


Ошибок на validation: 270


In [5]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / "predictions").mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, MODELS_DIR / "tfidf_logreg.joblib")

validation_predictions.to_csv(
    REPORTS_DIR / "predictions" / "tfidf_validation.csv", index=False
)

best_result = results_df.iloc[0].to_dict()

with open(REPORTS_DIR / "tfidf_summary.json", "w", encoding="utf-8") as file:
    json.dump(best_result, file, ensure_ascii=False, indent=2)

print("Лучшая конфигурация:", best_name)

Лучшая конфигурация: tfidf_unigrams


## Вывод

На текущем разбиении лучший baseline — TF-IDF с униграммами.
Его macro F1 используется как точка отсчёта для нейросетевых моделей.

Добавление биграмм не гарантирует улучшения: в коротких обращениях
они создают много редких признаков и могут ухудшить обобщение.

## Вывод

На validation униграммы дали macro F1 **0,8613**, униграммы с биграммами — **0,8395**. Число признаков при добавлении биграмм выросло с 1 357 до 8 859, а качество снизилось. Дальше использую более компактный вариант с униграммами.